# ApexPlanet Software Pvt. Ltd. — Data Analytics Internship
## Task 1: Data Immersion & Wrangling
**Dataset:** `ApexPlanet_DataAnalytics_Dataset.xlsx` (sheet: `Sales_Dataset`)\
**Prepared by:** Brojo Mohan Dutta\
**Environment:** Anaconda / Jupyter Notebook 7.4.5 / Python 3.x / Pandas

This notebook covers three steps: (1) initial familiarization, (2) data quality
profiling, and (3) an end-to-end cleaning & feature-engineering pipeline that
exports an analysis-ready CSV.

## Step 1 — Load & Familiarize

In [14]:
import pandas as pd
import numpy as np

RAW_PATH = "ApexPlanet_DataAnalytics_Dataset.xlsx"
df = pd.read_excel(RAW_PATH, sheet_name="Sales_Dataset")

print("Shape:", df.shape)
df.head()

Shape: (1000, 12)


,Order_ID,Order_Date,Customer_ID,Customer_Name,Age,Gender,City,Product,Category,Quantity,Unit_Price,Total_Sales
0,ORD100002,2025-02-25,CUST5529,Customer_227,30.0,Female,Bengaluru,Rice,Grocery,7,2829.77,19808.39
1,ORD100003,2025-10-14,CUST3127,Customer_182,63.0,Male,Bengaluru,Book,Education,5,27906.16,139530.80
2,ORD100004,2025-05-13,CUST8887,Customer_487,62.0,Female,Bengaluru,Book,Education,8,37491.06,299928.48
3,ORD100005,2025-12-02,CUST2515,Customer_470,65.0,Female,Kolkata,Mobile,Electronics,9,28541.36,256872.24
4,ORD100006,2025-11-20,CUST4796,Customer_380,44.0,Male,Bengaluru,Rice,Grocery,10,14036.59,140365.90


In [15]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1000 entries, 0 to 999
Data columns (total 12 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   Order_ID       1000 non-null   object 
 1   Order_Date     1000 non-null   object 
 2   Customer_ID    1000 non-null   object 
 3   Customer_Name  1000 non-null   object 
 4   Age            980 non-null    float64
 5   Gender         1000 non-null   object 
 6   City           987 non-null    object 
 7   Product        1000 non-null   object 
 8   Category       1000 non-null   object 
 9   Quantity       1000 non-null   int64  
 10  Unit_Price     1000 non-null   float64
 11  Total_Sales    1000 non-null   float64
dtypes: float64(3), int64(1), object(8)
memory usage: 93.9+ KB


## Step 2 — Data Quality Profiling

### 2.1 Missing values

In [16]:
missing = df.isnull().sum()
missing_pct = (missing / len(df) * 100).round(2)
pd.DataFrame({"Missing_Count": missing, "Missing_%": missing_pct}).query("Missing_Count > 0")

,Missing_Count,Missing_%
Age,20,2.0
City,13,1.3


### 2.2 Duplicate records

In [17]:
full_row_dupes = df.duplicated().sum()
order_id_dupes = df["Order_ID"].duplicated(keep=False).sum()

print(f"Full-row duplicates: {full_row_dupes}")
print(f"Rows sharing a non-unique Order_ID: {order_id_dupes}")

df[df["Order_ID"].duplicated(keep=False)].sort_values("Order_ID").head(9)

Full-row duplicates: 0
Rows sharing a non-unique Order_ID: 9


,Order_ID,Order_Date,Customer_ID,Customer_Name,Age,Gender,City,Product,Category,Quantity,Unit_Price,Total_Sales
48,ORD100050,2025-10-03,CUST5523,Customer_432,NaN,Male,Pune,Chair,Furniture,9,30834.87,277513.83
118,ORD100050,2025-01-22,CUST3194,Customer_114,34.0,Male,Kolkata,Book,Education,2,25795.55,51591.10
238,ORD100050,2025-03-01,CUST8467,Customer_104,23.0,Male,Bengaluru,Book,Education,3,2860.83,8582.49
358,ORD100050,2025-02-11,CUST6467,Customer_181,24.0,Female,Pune,Rice,Grocery,7,45480.39,318362.73
478,ORD100050,2025-11-22,CUST8225,Customer_468,64.0,Male,Bengaluru,Rice,Grocery,8,38957.59,311660.72
598,ORD100050,2025-02-21,CUST8859,Customer_53,NaN,Male,NaN,Laptop,Electronics,7,20144.84,141013.88
718,ORD100050,2025-11-06,CUST7824,Customer_190,31.0,Female,Mumbai,Rice,Grocery,4,42642.59,170570.36
838,ORD100050,2025-08-03,CUST5585,Customer_289,35.0,Male,Kolkata,Chair,Furniture,9,28747.47,258727.23
958,ORD100050,2025-02-09,CUST7188,Customer_89,62.0,Male,Bengaluru,Mobile,Electronics,1,34619.77,34619.77


### 2.3 Order_Date format check

In [18]:
print("Current dtype:", df["Order_Date"].dtype)
print(df["Order_Date"].head())

# Confirm every value matches YYYY-MM-DD before trusting a bulk parse
import re
pattern_check = df["Order_Date"].astype(str).apply(lambda x: re.sub(r"\d", "#", x))
pattern_check.value_counts()

Current dtype: object
0    2025-02-25
1    2025-10-14
2    2025-05-13
3    2025-12-02
4    2025-11-20
Name: Order_Date, dtype: object


Order_Date
####-##-##    1000
Name: count, dtype: int64

### 2.4 Outlier scan — Age, Quantity, Unit_Price, Total_Sales

In [19]:
numeric_cols = ["Age", "Quantity", "Unit_Price", "Total_Sales"]

def iqr_outliers(series):
    q1, q3 = series.quantile([0.25, 0.75])
    iqr = q3 - q1
    lower, upper = q1 - 1.5 * iqr, q3 + 1.5 * iqr
    return ((series < lower) | (series > upper)).sum()

for col in numeric_cols:
    print(f"{col}: {iqr_outliers(df[col].dropna())} IQR outliers | "
          f"min={df[col].min()}, max={df[col].max()}")

df[numeric_cols].describe()

Age: 0 IQR outliers | min=18.0, max=65.0
Quantity: 0 IQR outliers | min=1, max=10
Unit_Price: 0 IQR outliers | min=145.78, max=49997.53
Total_Sales: 19 IQR outliers | min=437.34, max=493677.5


,Age,Quantity,Unit_Price,Total_Sales
count,980.000000,1000.000000,1000.000000,1000.000000
mean,41.360204,5.435000,25486.783410,139399.439650
std,13.822597,2.838632,14179.402361,114100.051546
min,18.000000,1.000000,145.780000,437.340000
25%,30.000000,3.000000,13895.722500,47066.632500
50%,41.000000,5.000000,25398.740000,108594.025000
75%,54.000000,8.000000,37512.382500,203722.882500
max,65.000000,10.000000,49997.530000,493677.500000


**Findings summary**

| Issue | Detail |
|---|---|
| Missing values | `Age` — 20 nulls (2.0%); `City` — 13 nulls (1.3%) |
| Duplicates | 0 full-row duplicates; 9 rows share `Order_ID = ORD100050` with distinct customers/products/dates — a broken primary key, not a true duplicate |
| Date formatting | `Order_Date` loads as text (`object`/`str`), consistently `YYYY-MM-DD`, but is not yet a `datetime64` type |
| Outliers | No IQR outliers in `Age`, `Quantity`, `Unit_Price`, or `Total_Sales` — all values sit within plausible business ranges (Age 18–65, Quantity 1–10 units, Unit_Price ₹145.78–₹49,997.53) |


## Step 3 — Cleaning & Transformation Pipeline

### 3.1 Load dataset

In [20]:
RAW_PATH = "ApexPlanet_DataAnalytics_Dataset.xlsx"
df = pd.read_excel(RAW_PATH, sheet_name="Sales_Dataset")
print(f"Rows loaded: {df.shape[0]}, Columns loaded: {df.shape[1]}")

Rows loaded: 1000, Columns loaded: 12


### 3.2 Handle missing values

In [21]:
# Age (numeric, ~2% missing): median imputation preserves the distribution
# without being pulled by outliers the way a mean would be.
missing_age_before = df["Age"].isnull().sum()
df["Age"] = df["Age"].fillna(df["Age"].median())

# City (categorical, ~1.3% missing): explicit "Unknown" label rather than
# mode imputation — inventing a city would fabricate geographic signal
# that isn't in the source data.
missing_city_before = df["City"].isnull().sum()
df["City"] = df["City"].fillna("Unknown")

print(f"Age nulls filled: {missing_age_before} -> {df['Age'].isnull().sum()}")
print(f"City nulls filled: {missing_city_before} -> {df['City'].isnull().sum()}")

Age nulls filled: 20 -> 0
City nulls filled: 13 -> 0


### 3.3 Handle duplicates

In [22]:
# a) Full-row duplicates — safe to drop outright.
full_dupes = df.duplicated().sum()
df = df.drop_duplicates()

# b) Order_ID duplicates with different transaction details (e.g. ORD100050
#    appears 9 times attached to different customers/products/dates). These
#    are NOT the same transaction re-entered — dropping them would silently
#    delete real sales. They are a broken primary key. Fix: reassign a
#    guaranteed-unique surrogate ID and flag the affected rows for the
#    source-system owner instead of destroying the records.
dupe_id_mask = df["Order_ID"].duplicated(keep=False)
n_dupe_id_rows = dupe_id_mask.sum()

df = df.reset_index(drop=True)
df["Order_ID_Original"] = df["Order_ID"]
df["Order_ID_Flag"] = np.where(dupe_id_mask, "Duplicate_ID_Reassigned", "OK")
df.loc[dupe_id_mask, "Order_ID"] = [
    f"{oid}-{i+1:02d}" for i, oid in enumerate(df.loc[dupe_id_mask, "Order_ID"])
]

print(f"Full-row duplicates dropped: {full_dupes}")
print(f"Order_ID collisions repaired (surrogate suffix applied): {n_dupe_id_rows}")

Full-row duplicates dropped: 0
Order_ID collisions repaired (surrogate suffix applied): 9


### 3.4 Standardize `Order_Date` to datetime

In [23]:
df["Order_Date"] = pd.to_datetime(df["Order_Date"], format="%Y-%m-%d", errors="coerce")
unparseable_dates = df["Order_Date"].isnull().sum()
print(f"Order_Date converted to datetime64. Unparseable dates: {unparseable_dates}")
df["Order_Date"].head()

Order_Date converted to datetime64. Unparseable dates: 0


0   2025-02-25
1   2025-10-14
2   2025-05-13
3   2025-12-02
4   2025-11-20
Name: Order_Date, dtype: datetime64[ns]

### 3.5 Feature engineering

In [24]:
# Time-based features for trend/seasonality analysis
df["Order_Year"] = df["Order_Date"].dt.year
df["Order_Month"] = df["Order_Date"].dt.month
df["Order_Month_Name"] = df["Order_Date"].dt.month_name()

# Age_Group banding for demographic segmentation
age_bins = [17, 25, 35, 45, 55, 65]
age_labels = ["18-25", "26-35", "36-45", "46-55", "56-65"]
df["Age_Group"] = pd.cut(df["Age"], bins=age_bins, labels=age_labels, include_lowest=True)

# Revenue-per-unit sanity metric (useful QA + analysis field)
df["Avg_Price_Check"] = (df["Total_Sales"] / df["Quantity"]).round(2)

df[["Order_Date", "Order_Year", "Order_Month_Name", "Age", "Age_Group"]].head()

,Order_Date,Order_Year,Order_Month_Name,Age,Age_Group
0,2025-02-25,2025,February,30.0,26-35
1,2025-10-14,2025,October,63.0,56-65
2,2025-05-13,2025,May,62.0,56-65
3,2025-12-02,2025,December,65.0,56-65
4,2025-11-20,2025,November,44.0,36-45


### 3.6 Export analysis-ready dataset

In [25]:
OUTPUT_PATH = "cleaned_sales_dataset.csv"
df.to_csv(OUTPUT_PATH, index=False)
print(f"Cleaned dataset exported to: {OUTPUT_PATH}")
print(f"Final shape: {df.shape}")
df.head()

Cleaned dataset exported to: cleaned_sales_dataset.csv
Final shape: (1000, 19)


,Order_ID,Order_Date,Customer_ID,Customer_Name,Age,Gender,City,Product,Category,Quantity,Unit_Price,Total_Sales,Order_ID_Original,Order_ID_Flag,Order_Year,Order_Month,Order_Month_Name,Age_Group,Avg_Price_Check
0,ORD100002,2025-02-25,CUST5529,Customer_227,30.0,Female,Bengaluru,Rice,Grocery,7,2829.77,19808.39,ORD100002,OK,2025,2,February,26-35,2829.77
1,ORD100003,2025-10-14,CUST3127,Customer_182,63.0,Male,Bengaluru,Book,Education,5,27906.16,139530.80,ORD100003,OK,2025,10,October,56-65,27906.16
2,ORD100004,2025-05-13,CUST8887,Customer_487,62.0,Female,Bengaluru,Book,Education,8,37491.06,299928.48,ORD100004,OK,2025,5,May,56-65,37491.06
3,ORD100005,2025-12-02,CUST2515,Customer_470,65.0,Female,Kolkata,Mobile,Electronics,9,28541.36,256872.24,ORD100005,OK,2025,12,December,56-65,28541.36
4,ORD100006,2025-11-20,CUST4796,Customer_380,44.0,Male,Bengaluru,Rice,Grocery,10,14036.59,140365.90,ORD100006,OK,2025,11,November,36-45,14036.59
